In [1]:
import pandas as pd
import re

In [13]:
def clean_candidate(name):
    if not isinstance(name, str):
        return name

    prefixes = r'^(?:(?:Adv|Dr|Mr|Mrs|Ms|Shri|Smt|Prof)\.?\s+)+'
    name = re.sub(prefixes, "", name, flags=re.IGNORECASE)

    lineage_field = r'\s+(?:S/o|D/o|W/o|C/o|H/o|@)\.?\s+.*$'
    name = re.sub(lineage_field, "", name, flags=re.IGNORECASE)

    name = name.split(",", 1)[0]
    name = name.split("(", 1)[0]
    name = name.replace(".", " ")

    words = name.split()
    words = [word.capitalize() for word in words]
    return " ".join(words)

In [3]:
def clean_age(age):
    age = pd.to_numeric(age, errors="coerce")
    return int(age) if pd.notna(age) else -1

In [4]:
def clean_money(money):
    amount = re.search(r'(?<=Rs[.\s])([\d,]+)', money)
    if amount:
        amount = amount.group(1)
    else:
        return -1

    return int(amount.replace(",",""))

In [5]:
andhra_14 = pd.read_csv("ocr_data/andhra_2014.csv")
andhra_14.head()

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,Abburi Koteswara Rao,Pyramid Party of India,0,10th Pass,49,"Rs 1,76,20,000\r\n~ 1 Crore+","Rs 38,00,000\r\n~ 38 Lacs+"
1,2,Abdul Waheed Shaik,INC,0,10th Pass,44,"Rs 76,54,500\r\n~ 76 Lacs+","Rs 70,00,000\r\n~ 70 Lacs+"
2,3,Adala Prabhakara Reddy,TDP,0,12th Pass,64,"Rs 1,28,25,85,640\r\n~ 128 Crore+","Rs 13,16,46,350\r\n~ 13 Crore+"
3,4,Aithabathula Jogeswara Venkata Butchi Maheswar...,INC,1,Post Graduate,63,"Rs 1,26,25,844\r\n~ 1 Crore+","Rs 25,04,500\r\n~ 25 Lacs+"
4,5,Akki Reddy Bhimanadula,Pyramid Party of India,0,10th Pass,33,"Rs 6,44,500\r\n~ 6 Lacs+",Rs 0\r\n~


In [6]:
andhra_14.dtypes

Sno.              int64
Candidate           str
Party               str
Criminal Cases    int64
Education           str
Age                 str
Total Assets        str
Liabilities         str
dtype: object

In [7]:
andhra_14["Candidate"] = andhra_14["Candidate"].apply(clean_candidate)
andhra_14["Age"] = andhra_14["Age"].apply(clean_age)
andhra_14["Total Assets"] = andhra_14["Total Assets"].apply(clean_money)
andhra_14["Liabilities"] = andhra_14["Liabilities"].apply(clean_money)

In [8]:
andhra_14.dtypes

Sno.              int64
Candidate           str
Party               str
Criminal Cases    int64
Education           str
Age               int64
Total Assets      int64
Liabilities       int64
dtype: object

In [ ]:
# andhra_14.to_parquet("cleaned_data/andhra_2014.parquet", index=False)

In [ ]:
datasets = [
    ("andhra", 2019),
    ("bihar", 2014),
    ("bihar", 2019),
    ("gujarat", 2014),
    ("gujarat", 2019),
]

for state, year in datasets:
    df = pd.read_csv(f"ocr_data/{state}_{year}.csv")

    df["Candidate"] = df["Candidate"].apply(clean_candidate)
    df["Age"] = df["Age"].apply(clean_age)
    df["Total Assets"] = df["Total Assets"].apply(clean_money)
    df["Liabilities"] = df["Liabilities"].apply(clean_money)

    # df.to_parquet(
    #     f"cleaned_data/{state}_{year}.parquet",
    #     index=False
    # )